In [ ]:
import sys
import os
import pandas as pd
from collections import defaultdict
import numpy as np
from graphviz import Digraph

GET ICTV CLUSTERS

In [2]:
vfam_hits_filepath = "/p/lustre1/golez1/VOGDB_vfam_ICTV_ref_seqs/hits.tsv"
vog_hits_filepath = "/p/lustre1/golez1/VOGDB_vog_ICTV_ref_seqs/hits.tsv"

outdir = "/p/lustre1/golez1/VOGDB_clusters"

In [3]:
def read_hits_filepath(filepath):
    df = pd.read_csv(filepath, sep="\t", usecols=["QUERY", "FAM", "LINEAGE"])

    query_hit_map = dict()
    query_tax_map = dict()

    for i, row in df.iterrows():
        query = row["QUERY"]
        fam = row["FAM"]
        tax = row["LINEAGE"]

        query_hit_map[query] = fam
        query_tax_map[query] = tax
    
    return query_hit_map, query_tax_map

vfam_query_hit_map, vfam_query_tax_map = read_hits_filepath(vfam_hits_filepath)
vog_query_hit_map, vog_query_tax_map = read_hits_filepath(vog_hits_filepath)

In [4]:
print(len(set(vfam_query_hit_map.values())))
print(len(set(vog_query_hit_map.values())))

36252
54097


In [5]:
info = list()

all_queries = set(vfam_query_hit_map.keys()).union(set(vog_query_hit_map.keys()))

for query in all_queries:
    info.append(
        (
            query,
            vog_query_hit_map[query] if query in vog_query_hit_map.keys() else np.nan,
            vfam_query_hit_map[query] if query in vfam_query_hit_map.keys() else np.nan,
            vog_query_tax_map[query] if query in vog_query_tax_map.keys() else vfam_query_tax_map[query]
        )
    )

vog_df = pd.DataFrame(info, columns=["QUERY", "VOG", "vFAM", "TAXONOMY"]).sort_values(by=["vFAM", "VOG", "TAXONOMY"])

In [6]:
def remove_duplicated_vogs(vog_df):
    duplicated_vogs = set()

    for i, row in vog_df.iterrows():
        vog = row["VOG"]
        if vog not in duplicated_vogs:
            vfam_count = 0
            for vfam, group_df in vog_df.groupby("vFAM"):
                vfam_vogs = group_df["VOG"].tolist()
                
                if vog in vfam_vogs:
                    vfam_count += 1
                
                if vfam_count > 1:
                    print(vog)
                    duplicated_vogs.add(vog)
                    break
    
    vog_df = vog_df[~vog_df["VOG"].isin(duplicated_vogs)]

    return vog_df
    
# vog_df = remove_duplicated_vogs(vog_df)

In [7]:
# find_lca_taxa(all_taxa)
#
def find_lca_taxa(all_taxa):
    all_split_taxa = [taxa.split("; ") for taxa in all_taxa]
    lca_index = 0
    while True:
        if not all(lca_index < len(taxa) for taxa in all_split_taxa):
            break
        if not all(taxa[lca_index] == all_split_taxa[0][lca_index] for taxa in all_split_taxa):
            break
        lca_index += 1

    return "; ".join(all_split_taxa[0][:lca_index])

In [8]:
'''
n = 25
first_n_fams = vog_df['vFAM'].unique()[:n]

small_vog_df = vog_df[vog_df['vFAM'].isin(first_n_fams)]

added_taxa = set()

dot = Digraph(comment='Clusters Connecting to a Word')
dot.attr(rankdir='TB')

for vfam, group_df in small_vog_df.groupby("vFAM"):
    group_df = group_df.dropna()

    lca_taxa = find_lca_taxa(group_df["TAXONOMY"].tolist())
    
    if lca_taxa not in added_taxa:
        dot.node(lca_taxa, lca_taxa)
        added_taxa.add(lca_taxa)

    with dot.subgraph(name=f"cluster_{vfam}") as cluster:
        cluster.attr(label=vfam)

        for vog in set(group_df["VOG"].tolist()):
            cluster.node(vog, vog)

        dot.edge(vog, lca_taxa, ltail=f"cluster_{vfam}")

dot.save(os.path.join(outdir, "mygraph.dot"))
dot.render(os.path.join(outdir, 'example'), format='png', engine='fdp')
'''

'\nn = 25\nfirst_n_fams = vog_df[\'vFAM\'].unique()[:n]\n\nsmall_vog_df = vog_df[vog_df[\'vFAM\'].isin(first_n_fams)]\n\nadded_taxa = set()\n\ndot = Digraph(comment=\'Clusters Connecting to a Word\')\ndot.attr(rankdir=\'TB\')\n\nfor vfam, group_df in small_vog_df.groupby("vFAM"):\n    group_df = group_df.dropna()\n\n    lca_taxa = find_lca_taxa(group_df["TAXONOMY"].tolist())\n\n    if lca_taxa not in added_taxa:\n        dot.node(lca_taxa, lca_taxa)\n        added_taxa.add(lca_taxa)\n\n    with dot.subgraph(name=f"cluster_{vfam}") as cluster:\n        cluster.attr(label=vfam)\n\n        for vog in set(group_df["VOG"].tolist()):\n            cluster.node(vog, vog)\n\n        dot.edge(vog, lca_taxa, ltail=f"cluster_{vfam}")\n\ndot.save(os.path.join(outdir, "mygraph.dot"))\ndot.render(os.path.join(outdir, \'example\'), format=\'png\', engine=\'fdp\')\n'

In [9]:
vog_df.to_csv(os.path.join(outdir, "VOGDB_ICTV_df.tsv"), sep="\t", index=False)

GET ACTUAL CLUSTERS

In [10]:
vfam_members_filepath = "/p/vast1/ibap/chivian1/proj/viral_fams/dbs/VOGDB_r225/vfam/vfam.members.tsv"
vog_members_filepath = "/p/vast1/ibap/chivian1/proj/viral_fams/dbs/VOGDB_r225/vogs/vog.members.tsv"

In [27]:
def get_members(filepath):
    members_map = defaultdict(list)

    df = pd.read_csv(filepath, sep="\t", usecols=["#GroupName", "ProteinIDs"])

    for i, row in df.iterrows():
        group = row["#GroupName"]
        protein_ids = row["ProteinIDs"].split(",")
        members_map[group] = protein_ids

    return members_map

vfam_members_map = get_members(vfam_members_filepath)
vog_members_map = get_members(vog_members_filepath)

In [31]:
id_vogs_map = defaultdict(list)
for vog, vog_members in vog_members_map.items():
    for member in vog_members:
        id_vogs_map[member].append(vog)

vfam_vog_clusters = defaultdict(list)
for vfam, vfam_members in vfam_members_map.items():
    for vfam_member in vfam_members:
        for vog in id_vogs_map[vfam_member]:
            vfam_vog_clusters[vfam].append(vog)

In [32]:
with open(os.path.join(outdir, "actual_vfam_clusters.tsv"), "w") as file:
    for vfam, vogs in vfam_vog_clusters.items():
        file.write(f"{vfam}\t{','.join(sorted(list(set(vogs))))}\n")